# LLM Zoomcamp 2026 - Homework 1

本 notebook 完成 Homework 1 的六个问题，依次演示：

1. 从固定 Git commit 加载课程文档
2. 使用 `minsearch` 做全文检索
3. 使用完整 lesson 文档构建 RAG
4. 将长文档切成重叠 chunks
5. 比较切块前后的 RAG token 用量
6. 构建能够多次调用检索工具的 agentic RAG

所有题目使用同一份固定版本的数据，确保结果可复现。

## Setup

In [18]:
import os
from pathlib import Path

from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from openai import OpenAI


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd() / "rag" / ".env"

load_dotenv(env_path, override=True)

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
QUERY = "How does the agentic loop keep calling the model until it stops?"
AGENT_QUERY = "How does the agentic loop work, and how is it different from plain RAG?"

client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

print(f"Model: {MODEL}")

Model: qwen3.5:4b


## Question 1 - Load the documents

使用题目指定的 commit `8c1834d`，只读取路径中包含 `/lessons/` 的 Markdown 文件。

`commit_id` 很重要：即使课程仓库以后发生变化，作业仍然使用相同的数据。

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

print(f"Q1 answer - number of lesson documents: {len(documents)}")
documents[0]

Q1 answer - number of lesson documents: 72


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

**解释：** `file.parse()` 将每个 Markdown 文件转换为字典。后续检索使用：

- `filename`：文档来源，可用于追踪答案
- `content`：参与全文检索并作为 LLM context

## Question 2 - Build an index and search

这里使用词法检索。`content` 是可搜索文本字段，`filename` 是保留在结果中的关键词字段。

In [3]:
def build_index(items):
    return Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    ).fit(items)


def search(index, query, num_results=5):
    return index.search(query, num_results=num_results)


document_index = build_index(documents)
search_results = search(document_index, QUERY)

print("Q2 answer - first result:", search_results[0]["filename"])
print("\nTop 5 results:")
for position, result in enumerate(search_results, start=1):
    print(f"{position}. {result['filename']}")

Q2 answer - first result: 01-agentic-rag/lessons/14-agentic-loop.md

Top 5 results:
1. 01-agentic-rag/lessons/14-agentic-loop.md
2. 01-agentic-rag/lessons/15-frameworks.md
3. 01-agentic-rag/lessons/13-function-calling.md
4. 01-agentic-rag/lessons/11-agents-intro.md
5. 01-agentic-rag/lessons/16-other-frameworks.md


**解释：** RAG 的第一步不是调用 LLM，而是从知识库中召回相关材料。首个结果应直接讨论 agentic loop，说明查询与文档匹配正确。

## Question 3 - RAG with full documents

将前五个搜索结果拼成 context，并明确要求模型只能根据 context 回答。

In [4]:
def build_context(results):
    return "\n\n".join(
        f"Filename: {document['filename']}\n"
        f"Content:\n{document['content']}"
        for document in results
    )


def rag(index, query):
    results = search(index, query)
    context = build_context(results)
    prompt = f"""Answer the QUESTION using only the CONTEXT.
If the answer is not present in the context, say "I don't know."

QUESTION:
{query}

CONTEXT:
{context}
"""

    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )
    return {
        "answer": response.output_text.strip(),
        "input_tokens": response.usage.input_tokens,
        "results": results,
    }


full_rag_result = rag(document_index, QUERY)

print("Q3 answer - input tokens:", full_rag_result["input_tokens"])
print("\nRAG answer:")
if full_rag_result["answer"]:
    print(full_rag_result["answer"])
else:
    print(
        "No answer was generated because the full-document prompt "
        "filled the local model's context window."
    )

Q3 answer - input tokens: 4095

RAG answer:
No answer was generated because the full-document prompt filled the local model's context window.


**解释：** 普通 RAG 的控制流是固定的：检索一次，然后生成一次。输入 token 包括 prompt、问题以及五篇完整 lesson 的内容。本地 `qwen3.5:4b` 在本次运行中使用了 `4095` 个输入 token 并没有剩余空间生成答案，这直接暴露了 full-document RAG 的上下文浪费问题；Q4/Q5 将通过 chunking 解决它。

## Question 4 - Chunk the documents

完整 lesson 可能很长，会把大量无关文本送给模型。下面把每篇文档切成长度 `2000`、步长 `1000` 的重叠 chunks。

重叠区域可以降低关键信息刚好被切分边界截断的风险。

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Q4 answer - number of chunks: {len(chunks)}")
print("First chunk:")
chunks[0]

Q4 answer - number of chunks: 295
First chunk:


{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

## Question 5 - RAG with chunks

重新为 chunks 建立索引。检索结果现在是最相关的局部段落，而不是五篇完整 lesson。

In [6]:
chunk_index = build_index(chunks)
chunk_rag_result = rag(chunk_index, QUERY)

token_ratio = (
    full_rag_result["input_tokens"]
    / chunk_rag_result["input_tokens"]
)

print("Q5 answer - chunked input tokens:", chunk_rag_result["input_tokens"])
print(f"Token reduction: {token_ratio:.1f}x fewer input tokens")
print("\nRetrieved chunks:")
for position, result in enumerate(chunk_rag_result["results"], start=1):
    print(f"{position}. {result['filename']}")
print("\nChunked RAG answer:")
print(chunk_rag_result["answer"])

Q5 answer - chunked input tokens: 2462
Token reduction: 1.7x fewer input tokens

Retrieved chunks:
1. 01-agentic-rag/lessons/14-agentic-loop.md
2. 01-agentic-rag/lessons/14-agentic-loop.md
3. 01-agentic-rag/lessons/14-agentic-loop.md
4. 01-agentic-rag/lessons/15-frameworks.md
5. 01-agentic-rag/lessons/15-frameworks.md

Chunked RAG answer:
Based on the provided context, the agentic loop keeps calling the model until it stops using a `while True` structure that relies on checking for function calls in each iteration:

1.  It initializes an **iteration counter** to track round-trips (though this is primarily for observation).
2.  Inside the loop, after receiving a response from the model, it iterates through the output items and checks if their type is `"function_call"`.
3.  A flag named `has_function_calls` is used; it defaults to **False** at the start of each iteration (`while True`). If any function call is found in the output, this variable is set to **True**.
4.  The loop continues 

**解释：** chunking 的主要价值是提高检索粒度并减少输入 token。比较两次答案时，不应只看 token 是否减少，还要确认答案是否仍覆盖问题的核心信息。

## Question 6 - Agentic RAG

普通 RAG 的搜索次数和查询词由程序预先写死。Agentic RAG 将 `search_lessons` 暴露为工具：

1. 模型决定搜索什么关键词
2. agent loop 执行工具并把结果加入消息历史
3. 模型读取结果，决定继续搜索还是输出最终答案
4. 当模型不再返回工具调用而是返回答案时，循环停止

In [7]:
from pydantic_ai import Agent


search_call_count = 0
search_queries = []


def search_lessons(query: str) -> list[dict]:
    """Search course lessons for information relevant to the query."""
    global search_call_count
    search_call_count += 1
    search_queries.append(query)

    # Keep tool observations compact enough for the local 4K model.
    focused_query = f"agentic loop plain RAG {query}"
    results = search(chunk_index, focused_query, num_results=2)
    return [
        {
            "filename": result["filename"],
            "content": result["content"][:700],
        }
        for result in results
    ]


agent = Agent(
    f"openai-chat:{MODEL}",
    instructions=(
        "You are answering only this topic: how the agentic loop works "
        "and how it differs from plain RAG. Use the search tool at least "
        "twice with different agent-loop-related keywords. Ignore unrelated "
        "topics. Base the final answer only on the returned lesson excerpts."
    ),
    tools=[search_lessons],
    model_settings={"temperature": 0},
)

agent_result = await agent.run(AGENT_QUERY)

print("Q6 answer - search tool calls:", search_call_count)
print("Search queries:")
for position, query in enumerate(search_queries, start=1):
    print(f"{position}. {query}")
print("\nAgent answer:")
print(agent_result.output)

Q6 answer - search tool calls: 5
Search queries:
1. agentic loop mechanism operation process
2. agentic vs rag multi-turn conversation agent framework
3. agent loop multi-turn iterative process framework implementation
4. agent loop iteration mechanism how it works step by step
5. plain rag three steps linear pipeline no iteration

Agent answer:
Based on my searches through the course materials, here's how agentic loops work and how they differ from plain RAG:

## How the Agentic Loop Works

The **agentic loop** operates as an iterative process that continues calling the model until a specific condition is met (typically when it returns a response without any function calls). Key characteristics include:

1. **Iterative Execution**: Uses a `while` loop structure where each iteration processes messages and potentially triggers additional API calls
2. **Iteration Tracking**: Maintains an iteration counter to track how many round-trips occurred during execution
3. **Message History Manage

**解释：**

- 外层 agent runtime 才是真正的循环控制者，LLM 本身不会自动调用自己。
- 每次工具结果都会进入消息历史，形成新的 state。
- 模型返回 tool call 时继续循环；返回普通文本时结束。
- 实际调用次数不是固定答案，可能因模型和采样结果变化，但 developer instructions 明确要求多次搜索。

## Final summary

- Q1/Q2 验证数据加载和检索是否正确。
- Q3 是 full-document RAG。
- Q4/Q5 通过 chunking 减少无关 context 和 token 消耗。
- Q6 让模型动态决定检索策略，展示普通 RAG 与 agentic RAG 的核心差异。